# Healthcare Readmission Prediction
## 01 — Data Cleaning and Preprocessing

This notebook performs the first data-analysis stage for the Diabetes 130-US Hospitals dataset.

## Objectives
- Load the public UCI dataset
- Inspect rows, columns and data types
- Convert `?` to missing values
- Calculate missing-value counts and percentages
- Check duplicate records
- Generate numerical summaries
- Calculate the 30-day readmission rate
- Create a binary modelling target

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)


## Load the dataset
The dataset can be obtained from the UCI Machine Learning Repository using `ucimlrepo`.

In [ ]:
# Run this line once if the package is not installed:
# %pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

diabetes = fetch_ucirepo(id=296)
X = diabetes.data.features.copy()
y = diabetes.data.targets.copy()
df = pd.concat([X, y], axis=1)

print('Dataset loaded successfully')
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

## 1. Initial assessment

In [ ]:
print('Shape:', df.shape)
print('\nFirst five records:')
display(df.head())
print('\nData types:')
display(df.dtypes.value_counts())
print('\nDataset information:')
df.info()

## 2. Convert `?` to missing values

In [ ]:
question_marks = (df == '?').sum().sum()
print('Total ? placeholders before conversion:', int(question_marks))

df = df.replace('?', np.nan)
print('Total missing values after conversion:', int(df.isna().sum().sum()))

## 3. Missing-value summary

Missing percentage = missing observations / total observations × 100

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
}).sort_values('missing_percent', ascending=False)

display(missing_summary.head(15))

## 4. Duplicate records

In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_percent = duplicate_count / len(df) * 100

print('Duplicate rows:', duplicate_count)
print(f'Duplicate percentage: {duplicate_percent:.2f}%')

## 5. Numerical descriptive statistics

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns
numeric_summary = df[numeric_columns].describe().T
display(numeric_summary)

## 6. Readmission distribution and 30-day rate

In [ ]:
target_counts = df['readmitted'].value_counts(dropna=False)
display(target_counts)

early_readmissions = int(target_counts.get('<30', 0))
total_encounters = len(df)
rate = early_readmissions / total_encounters * 100

print(f'Early readmissions (<30 days): {early_readmissions:,}')
print(f'Total encounters: {total_encounters:,}')
print(f'30-day readmission rate: {rate:.2f}%')

## 7. Create a binary modelling target

`1` represents readmission within 30 days (`<30`). `0` represents `>30` or `NO`. The original `readmitted` column is retained.

In [ ]:
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

print('Binary target counts:')
display(df['readmitted_30d'].value_counts())

print('Binary target percentages:')
display((df['readmitted_30d'].value_counts(normalize=True) * 100).round(2))

## 8. Data-cleaning decision log

| Issue | Method | Reason |
|---|---|---|
| `?` placeholders | Convert to `NaN` | Treat unavailable information consistently |
| Missing values | Count and percentage | Identify variables needing review |
| Duplicates | Programmatic check | Prevent accidental duplicate observations |
| Numerical variables | Descriptive statistics | Understand range and distribution |
| Readmission target | Preserve original + create binary target | Support later classification modelling |

**Quality note:** Very high missingness should not automatically be replaced with a mean or mode. Each variable should be reviewed and the final treatment documented before modelling.

## 9. Final check

In [ ]:
print('Final rows:', len(df))
print('Final columns:', len(df.columns))
print('Total missing cells:', int(df.isna().sum().sum()))
print('Notebook data-cleaning assessment completed.')